quantum LDPCcode [[18,4,4]]

In [1]:
import numpy as np
from mip import Model, xsum, minimize, BINARY
from mip import OptimizationStatus
from bposd.css import css_code
from ldpc import bposd_decoder
import pickle
import itertools 
from scipy.sparse import coo_matrix, hstack 
from ldpc import mod2
from tabulate import tabulate
import matplotlib.pyplot as plt

In [2]:
import sys
import os
sys.path.append(os.path.abspath("..")) 
from functions_BB_code import *

## Code construction

In [3]:
# [[18, 4, 4]]
ell,m = 3, 3
a1,a2,a3 = 1, 0, 2
b1,b2,b3 = 1, 0, 2

# code length
n = 2*m*ell;  n2 = m*ell

I_ell = np.identity(ell,dtype=int) ; I_m = np.identity(m,dtype=int); I = np.identity(ell*m,dtype=int)
x = {} ;  y = {}

for i in range(ell):
	x[i] = np.kron(np.roll(I_ell,i,axis=1),I_m)
for i in range(m):
	y[i] = np.kron(I_ell,np.roll(I_m,i,axis=1))

A = (x[a1%ell] + y[a2%m] + y[a3%m]) % 2
B = (y[b1%m] + x[b2%ell] + x[b3%ell]) % 2

A1 = x[a1%ell]; A2 = y[a2%m]; A3 = y[a3%m]
B1 = y[b1%m]; B2 = x[b2%ell]; B3 = x[b3%ell]

AT = np.transpose(A) ; BT = np.transpose(B)

# Testing CSS code
hx = np.hstack((A,B));  hz = np.hstack((BT,AT))

remove_X_list = [2,8] ;  remove_Z_list = [3,4] ;

hx = np.delete(hx, remove_X_list, axis=0) ; hz = np.delete(hz, remove_Z_list, axis=0)

# number of logical qubits
k = n - rank2(hx) - rank2(hz)

qcode=css_code(hx=hx, hz=hz)
# qcode.compute_code_distance()  # this will take some time!
print('Testing CSS code...')
qcode.test()

Testing CSS code...
<Unnamed CSS code>, (3,6)-[[18,4,nan]]
 -Block dimensions: Pass
 -PCMs commute hz@hx.T==0: Pass
 -PCMs commute hx@hz.T==0: Pass
 -lx \in ker{hz} AND lz \in ker{hx}: Pass
 -lx and lz anticommute: Pass
 -<Unnamed CSS code> is a valid CSS code w/ params (3,6)-[[18,4,nan]]


True

In [4]:
# logical operator
lz = qcode.lz ;  lx = qcode.lx
# we choose a set of basis such that each logical X anticommute with its corresponding logical Z.
lz[1] = (lz[1]+lz[2]) %2 ; lz[3] = (lz[3]+lz[0]) %2

lz_copy = lz.copy()
lz_copy[0] = lz[1] ;  lz_copy[1] = lz[0] ;  lz_copy[2] = lz[3] ;  lz_copy[3] = lz[2] ;
lz = lz_copy
(lx@lz.T) %2

array([[1, 0, 0, 0],
       [0, 1, 0, 0],
       [0, 0, 1, 0],
       [0, 0, 0, 1]])

In [5]:
# Distance_test
print('Computing code distance...')
d = n
for i in range(k):
	w = distance_test(hz,lz[i,:])
	print('Logical qubit=',i,'Distance=',w)
	d = min(d,w)
    
for i in range(k):
	w = distance_test(hx,lx[i,:])
	print('Logical qubit=',i,'Distance=',w)
	d = min(d,w)
print('Code parameters: n,k,d=',n,k,d)

Computing code distance...
Logical qubit= 0 Distance= 4
Logical qubit= 1 Distance= 4
Logical qubit= 2 Distance= 4
Logical qubit= 3 Distance= 4
Logical qubit= 0 Distance= 4
Logical qubit= 1 Distance= 4
Logical qubit= 2 Distance= 4
Logical qubit= 3 Distance= 4
Code parameters: n,k,d= 18 4 4


In [6]:
# Connections of edges in the Tanner graph
lin_order, data_qubits, Xchecks, Zchecks, nbs = get_connection_Tanner(remove_X_list, remove_Z_list, n2)

In [7]:
# the order of two-qubit gates
sX = ['idle', 'idle', 1, 4, 3, 5, 0, 2] ; sZ = ['idle', 3, 5, 0, 1, 2, 4, 'idle'] ;

SM_cycle = get_SM_circuit_parallel(remove_X_list, remove_Z_list, lin_order, data_qubits, Xchecks, Zchecks, nbs, sX, sZ ) ;

In [8]:
cycle_append = []
for q in data_qubits:
    cycle_append.append(('final',q))

## Prior probability

In [9]:
# # depolarizing noise model 
from component_error_rates import *

# Setup BP-OSD decoder parameters
my_bp_method = "ms"
my_max_iter = 10000
my_osd_method = "osd_cs"
my_osd_order = 7
my_ms_scaling_factor = 0

## Decoder

In [10]:
Set_decoder_para_18_4_4 = {} ;

In [11]:
max_num_cycles = 6 ;

In [ ]:
for num_cycles in range(1, max_num_cycles + 1):

    # full syndrome measurement circuit
    cycle_repeated = (num_cycles-1) * SM_cycle[2*n2-len(remove_X_list)-len(remove_Z_list):] + \
                SM_cycle[2*n2-len(remove_X_list)-len(remove_Z_list) : -2*n2] + cycle_append  ;
    
    # Generating noisy circuits with a singe faulty operation
    X_circuits, X_Prob = get_Set_noisy_circuits_logical_X(cycle_repeated, error_rate_init, error_rate_idle, error_rate_H, error_rate_cz, \
                                                          error_rate_meas, error_final, error_DD_phaseflip, error_DD_bitflip)
    num_errX=len(X_circuits)
    print('Number of noisy circuits for the logical X state =',num_errX)
    
    
    Z_circuits, Z_Prob = get_Set_noisy_circuits_logical_Z(cycle_repeated, error_rate_init, error_rate_idle, error_rate_H, error_rate_cz, \
                                                          error_rate_meas, error_final, error_DD_phaseflip, error_DD_bitflip)
    num_errZ=len(Z_circuits)
    print('Number of noisy circuits for the logical Z state =',num_errZ)    
    
    channel_probsX, HX, HdecX, HXdict = decoding_X_matrix(X_circuits, X_Prob, num_cycles, SM_cycle, lin_order, n, k, data_qubits, \
                                                          Xchecks, lx, remove_X_list, remove_Z_list) ;
    
    channel_probsZ, HZ, HdecZ, HZdict = decoding_Z_matrix(Z_circuits, Z_Prob, num_cycles, SM_cycle, lin_order, n, k, data_qubits, \
                                                          Zchecks, lz, remove_X_list, remove_Z_list) ;
    
    Set_decoder_para_18_4_4[f"channel_probsX_{num_cycles}"] = channel_probsX ;
    Set_decoder_para_18_4_4[f"HX_{num_cycles}"] = HX ;
    Set_decoder_para_18_4_4[f"HdecX_{num_cycles}"] = HdecX ;

    Set_decoder_para_18_4_4[f"channel_probsZ_{num_cycles}"] = channel_probsZ ;
    Set_decoder_para_18_4_4[f"HZ_{num_cycles}"] = HZ ;
    Set_decoder_para_18_4_4[f"HdecZ_{num_cycles}"] = HdecZ ;  

In [ ]:
Set_decoder_para_18_4_4["hx"] = hx ;
Set_decoder_para_18_4_4["hz"] = hz ;
Set_decoder_para_18_4_4["lx"] = lx ;
Set_decoder_para_18_4_4["lz"] = lz ;

In [ ]:
with open('Set_decoder_para_18_4_4.pkl', 'wb') as f:
    pickle.dump(Set_decoder_para_18_4_4, f)